In [2]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType
import pickle , json
import numpy as np
import os
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv('../.env.example/.env')

True

In [3]:
# Best practice: store your credentials in environment variables
weaviate_url = os.environ["WEAVIATE_URL"]
weaviate_api_key = os.environ["WEAVIATE_API_KEY"]

In [4]:
embedding_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
enhanced_query = "sentence for armed robbery with deadly weapons"

In [9]:
query_embedding = embedding_model.encode([enhanced_query])
vector = query_embedding.tolist()
vector = vector[0]

In [10]:
with weaviate.connect_to_weaviate_cloud(
    cluster_url=weaviate_url,
    auth_credentials=weaviate_api_key,
) as client:

    # Step 2.2: Use this collection
    Euro_Laws = client.collections.use("Euro_Laws")

    # Step 2.3: Perform a vector search with NearVector
    response = Euro_Laws.query.near_vector(
        near_vector= vector , 
        limit=2
    )

    for obj in response.objects:
        print(json.dumps(obj.properties, indent=2))  # Inspect the results

{
  "subject_matter": "cooperation policy;  justice;  European construction;  European Union law;  criminal law",
  "act_name": "Council Framework Decision 2005/214/JHA of 24 February 2005 on the application of the principle of mutual recognition to financial penalties",
  "chunk_number": 7,
  "text": "were not carried out within the territory of the issuing State, the executing State may decide to reduce the amount of the penalty enforced to the maximum amount provided for acts of the same kind under the national law of the executing State, when the acts fall within the jurisdiction of that State. 2. The competent authority of the executing State shall, if necessary, convert the penalty into the currency of the executing State at the rate of exchange obtaining at the time when the penalty was imposed. Article 9 Law governing enforcement 1. Without prejudice to paragraph 3 of this Article, and to Article 10, the enforcement of the decision shall be governed by the law of the executing 

In [31]:
retrived = []
for obj in response.objects:
    retrived.append(json.dumps(obj.properties ))


In [52]:
celex_ids = []

In [ ]:

for meta in retrived:
         
    key_value = json.loads(meta)
    celex_ids.append(key_value.get("celex", ""))


celex_ids




['32016R1103', '32010R1259', '32016D1366', '32010R1259', '32010R1259']

# Metadata filter

In [1]:
from src.connection.clinets import weaviate_client
from langchain_weaviate import WeaviateVectorStore

In [2]:
properties = ['subject_matter']

In [5]:
Euro_Laws = weaviate_client.collections.use("Euro_Laws")

response = Euro_Laws.query.bm25(
    query='animal',
    query_properties=properties ,
    limit=5
)

WeaviateQueryError: Query call with protocol GRPC search failed with message Deadline Exceeded.